# Imports and data loading

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sympy.physics.control.control_plots import matplotlib
import matplotlib.ticker as ticker

In [2]:
%matplotlib tk

In [20]:
# --- Color Constants ---
GREEN = '\033[92m'
YELLOW = '\033[93m'
RED = '\033[91m'
ORANGE = '\033[38;5;208m'
RESET_COLORS = '\033[0m'

# Constants
DATASET_PATH = '../data/Dropbox_Dataset.npy'

# Load the raw string array
dataset_raw = np.load(DATASET_PATH, allow_pickle=True)

# Pre-processing: Split strings by comma and convert to DataFrame
# This fixes the Shape Mismatch and ValueError found earlier
dataset_split = [line.split(',') for line in dataset_raw]
df = pd.DataFrame(dataset_split, columns=['user', 'movie', 'rating', 'date'])

# Ensure rating is numeric for calculations
df['rating'] = pd.to_numeric(df['rating'], errors='coerce')

##  Initial Dataset Overview

In [4]:
print(f"DataFrame info:")
print(f"-Total ratings: {len(df)}")
# Using double quotes for f-strings to avoid SyntaxError
print(f"-Unique users in the initial dataframe: {df['user'].nunique()}")
print(f"-Unique movies in the initial dataframe: {df['movie'].nunique()}")
print('-'*50)

DataFrame info:
-Total ratings: 4669820
-Unique users in the initial dataframe: 1499238
-Unique movies in the initial dataframe: 351109
--------------------------------------------------


## User Rating Distribution Analysis
### Here we categorize users into logical bins based on how many ratings they have submitted to understand the distribution of user activity.

In [5]:
# Get the number of ratings per user
ratings_per_user = df.groupby('user').size()

# Slice the possible rating counts in logical, representative bins
bins = [0, 1, 2, 3, 10, 20, 100, float('inf')]
labels = ['1', '2', '3', '4-10', '11-20', '21-100', '100+']

# Put each user in the respective bin based on their ratings count
user_bins = pd.cut(ratings_per_user, bins=bins, labels=labels, right=True)

# Count number of users in each bin
bin_counts = user_bins.value_counts().sort_index()

# Print the results, raw and %
print(f'\nNumber of users in each rating count bin:')
print(bin_counts)
print(f'\nPercentage of users in each rating count bin:')
print((bin_counts / len(ratings_per_user) * 100).round(2))


Number of users in each rating count bin:
1         1069533
2          191934
3           76087
4-10       117959
11-20       24416
21-100      16156
100+         3153
Name: count, dtype: int64

Percentage of users in each rating count bin:
1         71.34
2         12.80
3          5.08
4-10       7.87
11-20      1.63
21-100     1.08
100+       0.21
Name: count, dtype: float64


In [6]:
plt.figure(figsize=(10, 6))
(bin_counts / 100000).plot(kind='bar')
plt.title('Number of Users by Rating Count')
plt.xlabel('Number of Ratings per User')
plt.ylabel('Number of Users (×100,000)')
plt.xticks(rotation=45)

plt.tight_layout()
plt.show()
print('-'*50)

--------------------------------------------------


## Analysis for Dimensionality Reduction
### This section explores "lonely ratings"—instances where a user has only one rating and that movie has also only been rated once—to evaluate potential filtering strategies.


In [7]:
# Count how many times each movie has been rated
movie_rating_counts = df['movie'].value_counts()

# Find movies rated exactly once
movies_rated_once = movie_rating_counts[movie_rating_counts == 1].index.to_list()
print(f"\nNumber of movies rated exactly once: {len(movies_rated_once)}")
print(f"Percentage of movies rated exactly once: {len(movies_rated_once) / len(movie_rating_counts) * 100:.2f}%")

# Find users with exactly one rating
one_rating_users = ratings_per_user[ratings_per_user == 1].index.to_list()
print(f"\nNumber of users with only one rating: {len(one_rating_users)}")

# Find the intersection ("lonely ratings")
lonely_ratings_mask = (df['user'].isin(one_rating_users)) & (df['movie'].isin(movies_rated_once))
lonely_ratings = df[lonely_ratings_mask]

print(f"\nNumber of ratings that are both the only rating for the user and the movie: \n {len(lonely_ratings)}")

# Check remaining movies if dropping 1-rating users
df_users_multiple_ratings = df[~df['user'].isin(one_rating_users)]
print(f"\nThe number of movies remaining if removing users with only one rating is:"
      f"\n{df_users_multiple_ratings['movie'].nunique()}")
print('-'*50)


Number of movies rated exactly once: 151175
Percentage of movies rated exactly once: 43.06%

Number of users with only one rating: 1069533

Number of ratings that are both the only rating for the user and the movie: 
 25342

The number of movies remaining if removing users with only one rating is:
319681
--------------------------------------------------


## Comparative Distribution: Original vs. Filtered Data
### We compare the dataset before and after removing movies that have only been rated once.

In [8]:
# User-ratings count comparison
df_movies_multiple_ratings = df[~df['movie'].isin(movies_rated_once)]

# FIX: Changed outer quotes to double quotes to avoid SyntaxError with df['user']
print(f"\nOriginal unique users: {df['user'].nunique()}")
print(f"Unique users after filtering: {df_movies_multiple_ratings['user'].nunique()}")

# Repeat calculation for filtered dataset
ratings_per_user_filtered = df_movies_multiple_ratings.groupby('user').size()
user_bins_filtered = pd.cut(ratings_per_user_filtered, bins=bins, labels=labels, right=True)
bin_counts_filtered = user_bins_filtered.value_counts().sort_index()

# Create comparison df
comparison_df = pd.DataFrame({
    'Original': bin_counts,
    'Filtered': bin_counts_filtered
}).fillna(0)

print('\nComparison of user counts:')
print(comparison_df)


Original unique users: 1499238
Unique users after filtering: 1473084

Comparison of user counts:
        Original  Filtered
1        1069533   1051259
2         191934    188568
3          76087     74657
4-10      117959    115897
11-20      24416     23869
21-100     16156     15794
100+        3153      3040


In [9]:
plt.figure(figsize=(12, 7))
x = np.arange(len(bin_counts.index))
width = 0.35

original_bars = plt.bar(x - width/2, bin_counts.values/100000, width, label='Original', color='skyblue', edgecolor='navy', alpha=0.7)
filtered_bars = plt.bar(x + width/2, bin_counts_filtered.reindex(bin_counts.index, fill_value=0)/100000, width, label='Filtered', color='lightcoral', edgecolor='darkred', alpha=0.7)

plt.xlabel('Number of Ratings per User', fontsize=12)
plt.ylabel('Number of Users (×100,000)', fontsize=12)
plt.title('Comparison: Users by Rating Count (Original vs Filtered)', fontsize=14, fontweight='bold')
plt.xticks(x, bin_counts.index, rotation=45)
plt.legend(fontsize=11)

plt.grid(axis='y', alpha=0.3, linestyle='--')
plt.tight_layout()
plt.show()
print('-'*50)

--------------------------------------------------


## Combined Filtering
### Removing both users with only one rating AND movies with only one rating to see the final dataset size.

In [10]:
# Remove users with only one rating AND movies rated only once
multiple_ratings_filter = (~df['user'].isin(one_rating_users)) & (~df['movie'].isin(movies_rated_once))
df_multiple_ratings = df[multiple_ratings_filter]

# Print info regarding the entries that are left after filtering
print(f'\nOriginal dataframe length (total entries): {len(df)}')
print(f'Final dataframe length after removing both conditions: {len(df_multiple_ratings)}')

# FIX: Using double quotes for the outer f-string to allow single quotes inside
print(f"\nWhat remains in the final filtered dataframe:")
print(f"Unique users: {df_multiple_ratings['user'].nunique()}")
print(f"Unique movies: {df_multiple_ratings['movie'].nunique()}")
print(f"Total ratings: {len(df_multiple_ratings)}")

print('-'*50)


Original dataframe length (total entries): 4669820
Final dataframe length after removing both conditions: 3474454

What remains in the final filtered dataframe:
Unique users: 428893
Unique movies: 193848
Total ratings: 3474454
--------------------------------------------------


## Movie Distribution Analysis
### Analyzing the distribution of movies based on the volume of ratings they have received.

In [11]:
movie_bins = [0, 1, 2, 3, 5, 10, 20, 50, 100, float('inf')]
movie_labels = ['1', '2', '3', '4-5', '6-10', '11-20', '21-50', '51-100', '100+']

movie_rating_bins = pd.cut(movie_rating_counts, bins=movie_bins, labels=movie_labels, right=True)
movie_bin_counts = movie_rating_bins.value_counts().sort_index()

print('\nNumber of movies by rating count:')
print(movie_bin_counts)

plt.figure(figsize=(10,6))
(movie_bin_counts / 1000).plot(kind='bar', color='mediumseagreen')
plt.title('Movie Distribution by Number of Ratings')
plt.xlabel('Number of Ratings per Movie')
plt.ylabel('Number of Movies (×1,000)')
plt.xticks(rotation=45)
plt.show()
print('-'*50)


Number of movies by rating count:
count
1         151175
2          53653
3          29312
4-5        32085
6-10       32183
11-20      21876
21-50      16844
51-100      6447
100+        7534
Name: count, dtype: int64
--------------------------------------------------


## Top N Movie Thresholding (Strategy 1)
### Keeping only the top 10,000 movies after removing users with only one rating.


In [12]:
# Pick a threshold for the number of movies we want to keep from the dataset (top n)
threshold = 10000

# Get the top n movies (based on number of ratings) on the df, after removing users with only 1 rating
movie_ratings_count_after_r1 = df_users_multiple_ratings['movie'].value_counts()
top_n_movies = movie_ratings_count_after_r1.nlargest(threshold).index

# Create a df with the entries regarding only those top n movies
df_top_movies = df_users_multiple_ratings[df_users_multiple_ratings['movie'].isin(top_n_movies)]

# Final counts
# FIX: Use double quotes for the f-string to allow single quotes inside for ['movie']
print(f"\nMovies kept after keeping top {threshold} movies: {df_top_movies['movie'].nunique()}")
print(f"Unique users remaining after keeping top {threshold} movies: {df_top_movies['user'].nunique()}")
print(f"Total ratings remaining after keeping top {threshold} movies: {len(df_top_movies)}")

ratings_per_movie = df_top_movies.groupby('movie').size()
print(f"The movies still remaining in the dataset contain at least {ratings_per_movie.min()} ratings.")


Movies kept after keeping top 10000 movies: 10000
Unique users remaining after keeping top 10000 movies: 380563
Total ratings remaining after keeping top 10000 movies: 2167588
The movies still remaining in the dataset contain at least 59 ratings.


## Top N Movie Thresholding (Strategy 2)
### Repeating the thresholding process, but strictly including users with at least 10 ratings.

In [13]:
n = 10
at_least_n_rating_users = ratings_per_user[ratings_per_user >= n].index.to_list()
df_users_filtered = df[df['user'].isin(at_least_n_rating_users)]

movie_ratings_count_after_r1_n = df_users_filtered['movie'].value_counts()
top_n_movies_second = movie_ratings_count_after_r1_n.nlargest(threshold).index
df_top_movies_second = df_users_filtered[df_users_filtered['movie'].isin(top_n_movies_second)]

print(f'\nMovies kept (users with ≥ {n} ratings): {df_top_movies_second["movie"].nunique()}')
print(f'Unique users remaining: {df_top_movies_second["user"].nunique()}')
print(f'Total ratings remaining: {len(df_top_movies_second)}')

ratings_per_movie_second = df_top_movies_second.groupby('movie').size()
print(f'The movies still remaining contain at least {ratings_per_movie_second.min()} ratings.')


Movies kept (users with ≥ 10 ratings): 10000
Unique users remaining: 48408
Total ratings remaining: 1342376
The movies still remaining contain at least 41 ratings.


## Rating distribution

In [18]:
# Calculate percentages
total_users_count = ratings_per_user.nunique()
activity_counts = focused_data.value_counts().sort_index()
activity_percentages = (activity_counts / len(ratings_per_user) * 100)

print(f"{YELLOW}Percentage of User Base per Rating Count:{RESET_COLORS}")
print("-" * 40)

for count, percent in activity_percentages.items():
    # Highlight users below the Trusted Threshold in Red
    color = RED if count < 3 else GREEN
    print(f"Users with {int(count):>2} rating(s): {color}{percent:>6.2f}%{RESET_COLORS}")

# Summary of the "Noise"
below_threshold = activity_percentages.loc[1:2].sum()
print("-" * 40)
print(f"{RED}Total users BELOW Trusted Threshold (1-2 ratings): {below_threshold:.2f}%{RESET_COLORS}")

Percentage of User Base per Rating Count:
----------------------------------------
Users with  1 rating(s): - 71.34%
Users with  2 rating(s): - 12.80%
Users with  3 rating(s): -  5.08%
Users with  4 rating(s): -  2.76%
Users with  5 rating(s): -  1.71%
Users with  6 rating(s): -  1.14%
Users with  7 rating(s): -  0.79%
Users with  8 rating(s): -  0.59%
Users with  9 rating(s): -  0.48%
Users with 10 rating(s): -  0.39%
Users with 11 rating(s): -  0.30%
Users with 12 rating(s): -  0.25%
Users with 13 rating(s): -  0.21%
Users with 14 rating(s): -  0.18%
Users with 15 rating(s): -  0.16%
Users with 16 rating(s): -  0.13%
Users with 17 rating(s): -  0.12%
Users with 18 rating(s): -  0.10%
Users with 19 rating(s): -  0.09%
Users with 20 rating(s): -  0.08%
----------------------------------------
Total users BELOW Trusted Threshold (1-2 ratings): 84.14%


In [25]:
# Set the dark background style
plt.style.use('dark_background')
plt.figure(figsize=(16, 8))

# Create the histogram with integer-centered bins
counts, bins, patches = plt.hist(focused_data, bins=np.arange(0.5, 21.5, 1),
                                 color='#6699AC', edgecolor='white', linewidth=0.3, alpha=0.9)

# Add Orange Labels with '-' sign on top of each bar
for i in range(len(counts)):
    pct = (counts[i] / len(ratings_per_user)) * 100
    if pct > 0.05:
        # Orange color and negative sign as requested
        plt.text(bins[i] + 0.5, counts[i] + (max(counts) * 0.01), f'-{pct:.1f}%',
                 ha='center', va='bottom', color='orange', fontsize=10, fontweight='bold')

# Keep the Trusted Threshold red line
plt.axvline(x=3, color='red', linestyle='--', linewidth=2, label='Trusted Threshold (3+)')

# Formatting and Styling
plt.title('User Activity Dispersity: Focus on 0-20 Ratings', fontsize=16, pad=25)
plt.xlabel('Number of Ratings Provided', fontsize=12)
plt.ylabel('Number of Users', fontsize=12)

# Ensure x-axis shows every integer
plt.xticks(range(1, 21))
plt.xlim(0.5, 20.5)

# Subtle gridlines
plt.grid(axis='y', linestyle=':', alpha=0.3)
plt.grid(axis='x', linestyle='-', alpha=0.1)

# Legend and spine cleanup
plt.legend(frameon=True, loc='upper right')
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

# Reset style
plt.style.use('default')